## Parte 1 — Softmax con temperatura

La softmax con temperatura asigna a cada token la probabilidad p_i = exp(z_i/T) / Σ_j exp(z_j/T). Para que el cálculo sea numéricamente estable en NumPy se resta el máximo de los logits antes de exponenciar: p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), con m = max_j z_j. El factor común exp(m/T) se cancela en el cociente, de modo que el resultado es idéntico al de la definición, pero los exponentes quedan acotados por arriba en 0 y no hay desbordamiento.

Aplicada a los logits t0=2.0, t1=1.0, t2=0.5, t3=0.2, t4=−1.0, t5=−3.0, las tres distribuciones (que suman 1.0 en los tres casos) y sus entropías en bits son:

| Token | T=0.5 | T=1.0 | T=2.0 |
|---|---|---|---|
| t0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.0225 | 0.0912 | 0.1457 |
| t4 | 0.0020 | 0.0275 | 0.0800 |
| t5 | 0.0000 | 0.0037 | 0.0294 |
| **Entropía (bits)** | **0.9147** | **1.7998** | **2.2888** |

El único valor que se muestra como 0.0000 es p(t5) con T=0.5, cuyo valor exacto es 3.7367e-05; al redondear a cuatro decimales se representa como 0.0000.

La entropía crece monótonamente con la temperatura: 0.9147 bits (T=0.5) < 1.7998 bits (T=1) < 2.2888 bits (T=2). Bajar T concentra la masa en t0 (su probabilidad sube de 0.5516 con T=1 a 0.8231 con T=0.5), mientras que subir T aplana la distribución (t0 baja a 0.3583 y el token menos probable, t5, sube de 0.0037 a 0.0294 al pasar de T=1 a T=2).

In [1]:
# T1: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
SUBTAREA T1 — Softmax con temperatura (NumPy, estable) y entropía en bits.
Tarea B — Temperatura, top-p y entropía de la distribución de salida (MMIA 6013).

Calcula la distribución softmax con temperatura para T = 0.5, 1 y 2 sobre los
logits dados (t0..t5), la entropía en bits de cada una, y presenta una tabla
con cuatro decimales. Guarda los resultados en resultados.json y una figura PNG.
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ----------------------------------------------------------------------
# Datos del enunciado
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


# ----------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable
# ----------------------------------------------------------------------
def softmax_con_temperatura(z, T):
    """
    p_i = exp(z_i / T) / sum_j exp(z_j / T)

    Estabilidad numérica: se resta el máximo de los logits escalados
    (z/T) antes de la exponencial, de modo que el mayor exponente es 0
    y no ocurre desbordamiento (overflow) en exp.
    """
    z = np.asarray(z, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura debe ser positiva.")
    escalados = z / T
    escalados = escalados - np.max(escalados)  # truco de estabilidad
    e = np.exp(escalados)
    return e / np.sum(e)


def entropia_bits(p):
    """H(p) = -sum p_i * log2(p_i), en bits (se omiten p_i = 0)."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


# ----------------------------------------------------------------------
# Cálculo de las tres distribuciones y sus entropías
# ----------------------------------------------------------------------
distribuciones = {}
entropias = {}
for T in temperaturas:
    p = softmax_con_temperatura(logits, T)
    distribuciones[T] = p
    entropias[T] = entropia_bits(p)

# Verificaciones del criterio de éxito
sumas = {f"T={T}": float(np.sum(distribuciones[T])) for T in temperaturas}
orden_entropia_ok = (entropias[0.5] < entropias[1.0]) and (entropias[1.0] < entropias[2.0])

# ----------------------------------------------------------------------
# Tabla con cuatro decimales
# ----------------------------------------------------------------------
tabla = pd.DataFrame(index=tokens)
for T in temperaturas:
    tabla[f"T={T}"] = [f"{p:.4f}" for p in distribuciones[T]]
tabla.loc["Entropía (bits)"] = {f"T={T}": f"{entropias[T]:.4f}" for T in temperaturas}
tabla.loc["Suma"] = {f"T={T}": f"{sumas[f'T={T}']:.4f}" for T in temperaturas}

print("=" * 60)
print("T1 — Softmax con temperatura y entropía (bits)")
print("=" * 60)
print("Logits:", dict(zip(tokens, logits)))
print()
print(tabla.to_string())
print()

# ----------------------------------------------------------------------
# Figura: distribuciones para las tres temperaturas
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(tokens))
ancho = 0.25
colores = ["#1f77b4", "#2ca02c", "#d62728"]
for i, T in enumerate(temperaturas):
    ax.bar(x + (i - 1) * ancho, distribuciones[T], width=ancho,
           label=f"T={T}  (H={entropias[T]:.4f} bits)", color=colores[i])
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylabel("Probabilidad")
ax.set_xlabel("Token")
ax.set_title("Softmax con temperatura — logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]")
ax.legend()
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
resultados = {
    "subtarea": "T1",
    "descripcion": "Softmax con temperatura (NumPy, numéricamente estable) y entropía en bits",
    "tokens": tokens,
    "logits": {t: float(z) for t, z in zip(tokens, logits)},
    "temperaturas": temperaturas,
    "distribuciones": {
        f"T={T}": {t: float(p) for t, p in zip(tokens, distribuciones[T])}
        for T in temperaturas
    },
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "sumas_distribuciones": sumas,
    "tabla_4_decimales": {
        "probabilidades": {
            f"T={T}": [f"{p:.4f}" for p in distribuciones[T]] for T in temperaturas
        },
        "entropias_bits": {f"T={T}": f"{entropias[T]:.4f}" for T in temperaturas},
    },
    "verificacion": {
        "distribuciones_suman_1": all(abs(s - 1.0) < 1e-12 for s in sumas.values()),
        "H_T0.5_menor_H_T1": bool(entropias[0.5] < entropias[1.0]),
        "H_T1_menor_H_T2": bool(entropias[1.0] < entropias[2.0]),
        "orden_entropias_correcto": bool(orden_entropia_ok),
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# Resumen
# ----------------------------------------------------------------------
print("Resumen:")
for T in temperaturas:
    print(f"  T={T}: H = {entropias[T]:.4f} bits | suma dist = {sumas[f'T={T}']:.6f}")
print(f"  Orden de entropías H(0.5) < H(1) < H(2): {orden_entropia_ok}")
print("Archivos generados: resultados.json, t1_softmax_temperatura.png")


T1 — Softmax con temperatura y entropía (bits)
Logits: {'t0': np.float64(2.0), 't1': np.float64(1.0), 't2': np.float64(0.5), 't3': np.float64(0.2), 't4': np.float64(-1.0), 't5': np.float64(-3.0)}

                  T=0.5   T=1.0   T=2.0
t0               0.8231  0.5516  0.3583
t1               0.1114  0.2029  0.2173
t2               0.0410  0.1231  0.1693
t3               0.0225  0.0912  0.1457
t4               0.0020  0.0275  0.0800
t5               0.0000  0.0037  0.0294
Entropía (bits)  0.9147  1.7998  2.2888
Suma             1.0000  1.0000  1.0000

Resumen:
  T=0.5: H = 0.9147 bits | suma dist = 1.000000
  T=1.0: H = 1.7998 bits | suma dist = 1.000000
  T=2.0: H = 2.2888 bits | suma dist = 1.000000
  Orden de entropías H(0.5) < H(1) < H(2): True
Archivos generados: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T=1 de la Parte 1 se ordenan los tokens de mayor a menor probabilidad y se acumula hasta alcanzar p=0.9:

| Orden | Token | Probabilidad | Acumulada | ¿Sobrevive? |
|---|---|---|---|---|
| 1 | t0 | 0.5516 | 0.5516 | sí |
| 2 | t1 | 0.2029 | 0.7546 | sí |
| 3 | t2 | 0.1231 | 0.8776 | sí |
| 4 | t3 | 0.0912 | 0.9688 | sí |
| 5 | t4 | 0.0275 | 0.9963 | no |
| 6 | t5 | 0.0037 | 1.0000 | no |

Los tres primeros tokens acumulan 0.8776, que no alcanza 0.9; hay que incluir t3, con el que la acumulada llega a 0.9688. El núcleo mínimo es por tanto {t0, t1, t2, t3} (4 tokens), que concentra una masa de 0.9688; quedan descartados t4 y t5, con una masa conjunta de 0.0312.

Renormalizando (dividiendo la probabilidad de cada superviviente entre 0.9688):

| Token | Probabilidad renormalizada |
|---|---|
| t0 | 0.5694 |
| t1 | 0.2095 |
| t2 | 0.1270 |
| t3 | 0.0941 |

La suma es 1.0. La renormalización reparte la masa descartada entre los supervivientes en proporción a su probabilidad original, por lo que todos suben ligeramente (t0 pasa de 0.5516 a 0.5694).

In [2]:
# T2: código aprobado por el crítico
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1 (Parte 1).

Pasos:
1. Carga la distribución de T=1 desde entrada/T1/resultados.json (subtarea previa)
   y la verifica recomputando la softmax con temperatura de forma numéricamente
   estable (si el archivo no existe, la recalcula desde los logits del enunciado).
2. Ordena los tokens por probabilidad descendente y calcula la probabilidad acumulada.
3. Conserva el conjunto MÍNIMO de tokens cuya acumulada alcanza p = 0.9
   (se incluye el token que cruza el umbral).
4. Renormaliza las probabilidades de los supervivientes (debe sumar 1).
5. Escribe resultados.json (contrato) y una figura PNG; imprime un resumen.
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Parámetros de la subtarea
# ----------------------------------------------------------------------
P = 0.9            # umbral top-p
TEMPERATURA = 1.0  # distribución de la Parte 1
RUTA_T1 = Path("entrada/T1/resultados.json")
LOGITS_ENUNCIADO = {"t0": 2.0, "t1": 1.0, "t2": 0.5, "t3": 0.2, "t4": -1.0, "t5": -3.0}


def softmax_con_temperatura(logits, T):
    """Softmax con temperatura, numéricamente estable (resta el máximo)."""
    z = np.asarray(logits, dtype=float) / float(T)
    z = z - z.max()
    e = np.exp(z)
    return e / e.sum()


# ----------------------------------------------------------------------
# 1. Distribución de T=1 (de la subtarea T1, con verificación)
# ----------------------------------------------------------------------
if RUTA_T1.exists():
    with open(RUTA_T1, "r", encoding="utf-8") as f:
        res_t1 = json.load(f)
    tokens = list(res_t1["tokens"])
    logits = np.array([res_t1["logits"][t] for t in tokens], dtype=float)
    p_t1 = np.array([res_t1["distribuciones"]["T=1.0"][t] for t in tokens], dtype=float)
    fuente = str(RUTA_T1)
    coincide_softmax = bool(
        np.allclose(p_t1, softmax_con_temperatura(logits, TEMPERATURA), atol=1e-9)
    )
else:
    tokens = list(LOGITS_ENUNCIADO.keys())
    logits = np.array(list(LOGITS_ENUNCIADO.values()), dtype=float)
    p_t1 = softmax_con_temperatura(logits, TEMPERATURA)
    fuente = "softmax T=1 recalculada desde los logits del enunciado"
    coincide_softmax = True

# ----------------------------------------------------------------------
# 2. Orden descendente y probabilidad acumulada
# ----------------------------------------------------------------------
orden = np.argsort(-p_t1, kind="stable")
tokens_ordenados = [tokens[i] for i in orden]
probs_ordenadas = p_t1[orden]
acumuladas = np.cumsum(probs_ordenadas)

# ----------------------------------------------------------------------
# 3. Corte top-p: conjunto mínimo con acumulada >= p
# ----------------------------------------------------------------------
idx_corte = int(np.searchsorted(acumuladas, P, side="left"))
if idx_corte >= len(acumuladas):  # salvaguarda (p < suma total = 1)
    idx_corte = len(acumuladas) - 1
n_sup = idx_corte + 1

tokens_sup = tokens_ordenados[:n_sup]
probs_sup = probs_ordenadas[:n_sup]
masa_sup = float(probs_sup.sum())

# ----------------------------------------------------------------------
# 4. Renormalización de los supervivientes
# ----------------------------------------------------------------------
p_renorm = probs_sup / masa_sup
suma_renorm = float(p_renorm.sum())

tabla_ordenada = [
    {
        "token": tokens_ordenados[i],
        "probabilidad": float(probs_ordenadas[i]),
        "acumulada": float(acumuladas[i]),
        "sobrevive": i < n_sup,
    }
    for i in range(len(tokens_ordenados))
]
distribucion_renormalizada = {
    tokens_sup[i]: float(p_renorm[i]) for i in range(n_sup)
}

# ----------------------------------------------------------------------
# 5. Contrato: resultados.json
# ----------------------------------------------------------------------
resultados = {
    "subtarea": "T2",
    "descripcion": "Muestreo de núcleo (top-p) con p=0.9 sobre la distribución de T=1",
    "parametros": {"p": P, "temperatura": TEMPERATURA},
    "fuente_distribucion": fuente,
    "distribucion_T1_original": {t: float(p_t1[i]) for i, t in enumerate(tokens)},
    "orden_descendente": {
        "tokens": tokens_ordenados,
        "probabilidades": [float(x) for x in probs_ordenadas],
        "probabilidades_acumuladas": [float(x) for x in acumuladas],
    },
    "tabla_ordenada": tabla_ordenada,
    "corte_top_p": {
        "p": P,
        "indice_corte": idx_corte,
        "num_supervivientes": n_sup,
        "tokens_supervivientes": tokens_sup,
        "tokens_descartados": tokens_ordenados[n_sup:],
        "masa_probabilidad_supervivientes": masa_sup,
        "masa_descartada": float(1.0 - masa_sup),
    },
    "distribucion_renormalizada": distribucion_renormalizada,
    "suma_distribucion_renormalizada": suma_renorm,
    "verificacion": {
        "coincide_con_softmax_recalculada": coincide_softmax,
        "suma_renormalizada_es_1": bool(np.isclose(suma_renorm, 1.0)),
        "acumulada_ultimo_superviviente_alcanza_p": bool(acumuladas[idx_corte] >= P),
        "acumulada_anterior_menor_que_p": bool(
            idx_corte == 0 or acumuladas[idx_corte - 1] < P
        ),
    },
}
resultados["verificacion"]["conjunto_es_minimo"] = bool(
    resultados["verificacion"]["acumulada_ultimo_superviviente_alcanza_p"]
    and resultados["verificacion"]["acumulada_anterior_menor_que_p"]
)

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

# ----------------------------------------------------------------------
# 6. Figura: probs. ordenadas + acumulada + corte, y renormalizada
# ----------------------------------------------------------------------
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))

ax = ejes[0]
x = np.arange(len(tokens_ordenados))
barras = ax.bar(x, probs_ordenadas, color="#4C72B0", label="p_i (T=1)")
for i in range(n_sup, len(x)):
    barras[i].set_color("#C44E52")
    barras[i].set_alpha(0.45)
ax2 = ax.twinx()
ax2.plot(x, acumuladas, "o-", color="#55A868", label="acumulada")
ax2.axhline(P, color="k", ls="--", lw=1, label=f"p = {P}")
ax2.set_ylim(0, 1.05)
ax.set_xticks(x)
ax.set_xticklabels(tokens_ordenados)
ax.set_ylabel("probabilidad")
ax2.set_ylabel("probabilidad acumulada")
ax.set_title("Top-p (p=0.9): probabilidades ordenadas y acumulada")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="center right", fontsize=8)

ax = ejes[1]
xs = np.arange(n_sup)
ax.bar(xs, p_renorm, color="#55A868")
ax.set_xticks(xs)
ax.set_xticklabels(tokens_sup)
ax.set_ylim(0, max(p_renorm) * 1.15)
ax.set_ylabel("probabilidad renormalizada")
ax.set_title(f"Distribución renormalizada (suma = {suma_renorm:.6f})")
for i, v in enumerate(p_renorm):
    ax.text(i, v + 0.008, f"{v:.4f}", ha="center", fontsize=8)

fig.tight_layout()
fig.savefig("T2_top_p_renormalizacion.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 7. Resumen
# ----------------------------------------------------------------------
print("T2 — Top-p (nucleus sampling) con p=0.9 sobre la distribución de T=1")
print("-" * 70)
print(f"{'token':<6}{'probabilidad':>16}{'acumulada':>14}{'sobrevive':>12}")
for fila in tabla_ordenada:
    print(
        f"{fila['token']:<6}{fila['probabilidad']:>16.10f}"
        f"{fila['acumulada']:>14.6f}{str(fila['sobrevive']):>12}"
    )
print("-" * 70)
print(f"Supervivientes (conjunto mínimo con acumulada >= {P}): {tokens_sup}")
print(f"Masa de probabilidad de los supervivientes: {masa_sup:.10f}")
print(f"Tokens descartados: {tokens_ordenados[n_sup:]} (masa descartada {1.0 - masa_sup:.10f})")
print("Distribución renormalizada:")
for t, v in distribucion_renormalizada.items():
    print(f"  {t}: {v:.10f}")
print(f"Suma de la distribución renormalizada: {suma_renorm:.12f}")
print("Figura guardada: T2_top_p_renormalizacion.png")
print("Resultados escritos en resultados.json")


T2 — Top-p (nucleus sampling) con p=0.9 sobre la distribución de T=1
----------------------------------------------------------------------
token     probabilidad     acumulada   sobrevive
t0        0.5516226379      0.551623        True
t1        0.2029306278      0.754553        True
t2        0.1230836475      0.877637        True
t3        0.0911826088      0.968820        True
t4        0.0274636740      0.996283       False
t5        0.0037168041      1.000000       False
----------------------------------------------------------------------
Supervivientes (conjunto mínimo con acumulada >= 0.9): ['t0', 't1', 't2', 't3']
Masa de probabilidad de los supervivientes: 0.9688195219
Tokens descartados: ['t4', 't5'] (masa descartada 0.0311804781)
Distribución renormalizada:
  t0: 0.5693760555
  t1: 0.2094617451
  t2: 0.1270449704
  t3: 0.0941172290
Suma de la distribución renormalizada: 1.000000000000
Figura guardada: T2_top_p_renormalizacion.png
Resultados escritos en resultados.json


## Parte 3 — Comprobación empírica

Se muestrearon 10 000 tokens de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Los conteos observados fueron t0=5682, t1=2093, t2=1296 y t3=929 (suman 10 000). El gráfico de barras de la celda siguiente compara las frecuencias observadas con las probabilidades teóricas:

| Token | Frecuencia observada | Probabilidad teórica | Diferencia (obs. − teórica) |
|---|---|---|---|
| t0 | 0.5682 | 0.5694 | −0.0012 |
| t1 | 0.2093 | 0.2095 | −0.0002 |
| t2 | 0.1296 | 0.1270 | 0.0026 |
| t3 | 0.0929 | 0.0941 | −0.0012 |

La divergencia KL de la distribución empírica respecto de la teórica, KL(q‖p) = Σ_i q_i · log2(q_i/p_i) sobre los tokens supervivientes, es de 5.0069e-05 bits. Un valor tan próximo a 0 confirma que el muestreo con la semilla fijada reproduce la distribución teórica: las frecuencias observadas difieren de las teóricas solo por el ruido estadístico esperable con n=10 000 (la mayor desviación absoluta es 0.0026, en t2).

In [3]:
# T3: código aprobado por el crítico
# Subtarea T3: Comprobación empírica de la distribución renormalizada (top-p p=0.9, T=1)
# - 10,000 muestras con numpy.random.default_rng(0)
# - Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# - Divergencia KL (empírica || teórica) en bits

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. Cargar la distribución renormalizada de la Parte 2 (entrada/T2)
# ------------------------------------------------------------------
ruta_t2 = Path("entrada/T2/resultados.json")
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        t2 = json.load(f)
    dist_renorm = t2["distribucion_renormalizada"]
    tokens = list(dist_renorm.keys())
    p_teorica = np.array([dist_renorm[t] for t in tokens], dtype=float)
    fuente = "entrada/T2/resultados.json"
else:
    # Respaldo: recalcular softmax(T=1) + top-p (p=0.9) a partir de los logits
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    tokens_todos = ["t0", "t1", "t2", "t3", "t4", "t5"]
    z = logits - logits.max()
    p_full = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p_full)
    acum = np.cumsum(p_full[orden])
    k = int(np.searchsorted(acum, 0.9) + 1)  # conjunto mínimo que alcanza 0.9
    supervivientes = np.sort(orden[:k])
    tokens = [tokens_todos[i] for i in supervivientes]
    p_teorica = p_full[supervivientes] / p_full[supervivientes].sum()
    fuente = "recalculada desde logits (softmax T=1 + top-p 0.9)"

p_teorica = p_teorica / p_teorica.sum()  # seguridad numérica

# ------------------------------------------------------------------
# 2. Tomar 10,000 muestras con numpy.random.default_rng(0)
# ------------------------------------------------------------------
rng = np.random.default_rng(0)
n_muestras = 10000
indices = rng.choice(len(tokens), size=n_muestras, p=p_teorica)

# ------------------------------------------------------------------
# 3. Distribución empírica (frecuencias observadas)
# ------------------------------------------------------------------
conteos = np.bincount(indices, minlength=len(tokens))
frecuencias = conteos / n_muestras

# ------------------------------------------------------------------
# 4. Divergencia KL en bits: KL(empírica || teórica) = Σ q_i log2(q_i / p_i)
# ------------------------------------------------------------------
mask = frecuencias > 0
kl_bits = float(np.sum(frecuencias[mask] * np.log2(frecuencias[mask] / p_teorica[mask])))

# ------------------------------------------------------------------
# 5. Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# ------------------------------------------------------------------
x = np.arange(len(tokens))
ancho = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - ancho / 2, frecuencias, ancho,
            label="Frecuencia observada (empírica)", color="#4C72B0")
b2 = ax.bar(x + ancho / 2, p_teorica, ancho,
            label="Probabilidad teórica (renormalizada)", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / frecuencia")
ax.set_title("Parte 3 — Comprobación empírica\n"
             "Top-p (p=0.9, T=1): 10,000 muestras, numpy.random.default_rng(0)")
ymax = max(frecuencias.max(), p_teorica.max())
ax.set_ylim(0, ymax * 1.18)
for barra, valor in list(zip(b1, frecuencias)) + list(zip(b2, p_teorica)):
    ax.text(barra.get_x() + barra.get_width() / 2, valor + ymax * 0.012,
            f"{valor:.4f}", ha="center", va="bottom", fontsize=8)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("parte3_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# ------------------------------------------------------------------
# 6. Escribir resultados.json (contrato de la subtarea)
# ------------------------------------------------------------------
resultados = {
    "subtarea": "T3",
    "descripcion": ("Comprobación empírica: 10,000 muestras de la distribución renormalizada "
                    "de la Parte 2 (top-p p=0.9 sobre softmax T=1), comparación de frecuencias "
                    "observadas vs probabilidades teóricas y divergencia KL en bits"),
    "parametros": {
        "n_muestras": n_muestras,
        "semilla": 0,
        "generador": "numpy.random.default_rng(0)",
        "p_top_p": 0.9,
        "temperatura": 1.0,
    },
    "fuente_distribucion": fuente,
    "tokens": tokens,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens, p_teorica)},
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(fr) for t, fr in zip(tokens, frecuencias)},
    "tabla_comparativa": [
        {
            "token": t,
            "frecuencia_observada": float(fr),
            "probabilidad_teorica": float(p),
            "diferencia_observada_menos_teorica": float(fr - p),
        }
        for t, fr, p in zip(tokens, frecuencias, p_teorica)
    ],
    "divergencia_kl_bits": kl_bits,
    "definicion_kl": "KL(empírica || teórica) = Σ q_i * log2(q_i / p_i), sobre los tokens supervivientes",
    "figura": "parte3_frecuencias_vs_teoricas.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------------
# 7. Resumen
# ------------------------------------------------------------------
print("=== Subtarea T3 — Comprobación empírica ===")
print(f"Fuente de la distribución: {fuente}")
print(f"Muestras: {n_muestras} con numpy.random.default_rng(0)")
print(f"Tokens supervivientes (top-p p=0.9): {tokens}")
print("Token | frecuencia observada | prob. teórica | diferencia")
for t, fr, p in zip(tokens, frecuencias, p_teorica):
    print(f"  {t}  |  {fr:.6f}  |  {p:.6f}  |  {fr - p:+.6f}")
print(f"Divergencia KL (empírica || teórica) en bits: {kl_bits:.8f}")
print("Figura guardada: parte3_frecuencias_vs_teoricas.png")
print("Resultados escritos en resultados.json")


=== Subtarea T3 — Comprobación empírica ===
Fuente de la distribución: entrada/T2/resultados.json
Muestras: 10000 con numpy.random.default_rng(0)
Tokens supervivientes (top-p p=0.9): ['t0', 't1', 't2', 't3']
Token | frecuencia observada | prob. teórica | diferencia
  t0  |  0.568200  |  0.569376  |  -0.001176
  t1  |  0.209300  |  0.209462  |  -0.000162
  t2  |  0.129600  |  0.127045  |  +0.002555
  t3  |  0.092900  |  0.094117  |  -0.001217
Divergencia KL (empírica || teórica) en bits: 0.00005007
Figura guardada: parte3_frecuencias_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

**T → 0 (concentración en el máximo).** Dividir los logits entre una temperatura muy pequeña amplifica sus diferencias: para cualquier par de tokens con z_i > z_j, el cociente exp((z_i − z_j)/T) crece sin cota cuando T disminuye. Toda la masa converge al token de mayor logit, t0 (z=2.0): p(t0) → 1, las demás probabilidades → 0 y la entropía → 0 bits. Los datos de la Parte 1 muestran esa tendencia: al bajar T de 1 a 0.5, p(t0) sube de 0.5516 a 0.8231, p(t5) cae a 3.7367e-05 y la entropía baja de 1.7998 a 0.9147 bits.

**T → ∞ (aplanamiento hacia la uniforme).** Con T muy grande, z_i/T → 0 para todo token, de modo que exp(z_i/T) → 1 y todas las probabilidades convergen al mismo valor: la distribución tiende a la uniforme sobre los seis tokens y la entropía tiende a su máximo posible. La serie medida va en esa dirección: con T=2 la distribución ya está visiblemente más plana (t0 baja a 0.3583 y t5 sube a 0.0294) y la entropía sube a 2.2888 bits, por encima de los 1.7998 bits de T=1 y los 0.9147 bits de T=0.5.

**Por qué T → 0 equivale a la decodificación voraz.** La decodificación voraz (greedy) selecciona en cada paso argmax_i z_i, que aquí es t0. Cuando T → 0, la softmax concentra toda la probabilidad en ese mismo token, de modo que una muestra extraída de esa distribución es t0 con probabilidad 1: muestrear de la distribución y tomar el argmax de los logits dejan de ser procedimientos distintos. La evidencia son las entropías de la Parte 1, que decrecen monótonamente al bajar T (2.2888 → 1.7998 → 0.9147 bits): la distribución pierde incertidumbre de forma continua y, en el límite T → 0, entropía cero significa que no queda azar que resolver, exactamente el comportamiento determinista de greedy.